# 05 — Feature Engineering
**Fingo Income Predictor** | Tim CC26-PSU217

Input: `data/synthetic/synthetic_52week_user_income.csv` + `data/processed/survey_temporal_mapped.csv`  
Output: `data/processed/income_features.csv`

**Income ordering:** `income_w4 (terlama) → income_w3 → income_w2 → income_w1 (terbaru)`

Notebook ini mengubah synthetic 52-week income menjadi supervised forecasting dataset memakai sliding window 4 minggu. Target utama adalah `next_week_income` dan target klasifikasi arah adalah `next_week_direction`.

In [1]:
# GIT PULL — Sinkronisasi terbaru dari remote sebelum mulai
import os, shutil, subprocess

try:
    from google.colab import userdata
except Exception:
    userdata = None

os.chdir("/content")

GITHUB_USERNAME = "ClarisyaA"
REPO_NAME       = "fingo-income-analysis"
BRANCH_NAME     = "feat/income-predictor-final"
LOCAL_DIR       = f"/content/{REPO_NAME}"
FRESH_CLONE     = False  # Set True hanya kalau mau clone ulang dari nol

def get_remote_url():
    try:
        token = userdata.get("GITHUB_TOKEN") if userdata else os.environ.get("GITHUB_TOKEN", "")
        if token:
            return f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git", token
    except Exception:
        pass
    return f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git", None

remote_url, token = get_remote_url()

def mask_cmd(cmd):
    return cmd.replace(token, "***TOKEN***") if token else cmd

def run_cmd(cmd, check=True, cwd="/content"):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    print(f"$ {mask_cmd(cmd)}")
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f"Command gagal: {mask_cmd(cmd)}")
    return r

def remote_branch_exists():
    r = run_cmd(f"git ls-remote --heads {remote_url} {BRANCH_NAME}", check=False)
    return r.stdout.strip() != ""

branch_exists = remote_branch_exists()

if FRESH_CLONE and os.path.exists(LOCAL_DIR):
    os.chdir("/content")
    shutil.rmtree(LOCAL_DIR)

if not os.path.exists(LOCAL_DIR):
    if branch_exists:
        run_cmd(f"git clone -b {BRANCH_NAME} {remote_url} {LOCAL_DIR}")
    else:
        run_cmd(f"git clone {remote_url} {LOCAL_DIR}")
        run_cmd(f"git checkout -b {BRANCH_NAME}", cwd=LOCAL_DIR)
else:
    run_cmd(f"git remote set-url origin {remote_url}", cwd=LOCAL_DIR)
    run_cmd("git fetch origin", cwd=LOCAL_DIR)
    if branch_exists:
        local_b = run_cmd(f"git branch --list {BRANCH_NAME}", check=False, cwd=LOCAL_DIR).stdout.strip()
        if local_b:
            run_cmd(f"git checkout {BRANCH_NAME}", cwd=LOCAL_DIR)
        else:
            run_cmd(f"git checkout -b {BRANCH_NAME} origin/{BRANCH_NAME}", cwd=LOCAL_DIR)
        run_cmd(f"git pull --rebase origin {BRANCH_NAME}", cwd=LOCAL_DIR)
    else:
        current_branch = run_cmd("git branch --show-current", check=False, cwd=LOCAL_DIR).stdout.strip()
        if current_branch != BRANCH_NAME:
            local_b = run_cmd(f"git branch --list {BRANCH_NAME}", check=False, cwd=LOCAL_DIR).stdout.strip()
            if local_b:
                run_cmd(f"git checkout {BRANCH_NAME}", cwd=LOCAL_DIR)
            else:
                run_cmd(f"git checkout -b {BRANCH_NAME}", cwd=LOCAL_DIR)

os.chdir(LOCAL_DIR)
run_cmd(f"git remote set-url origin {remote_url}", cwd=LOCAL_DIR)
print("\nRepo siap digunakan")
print(f"Working directory: {os.getcwd()}")
run_cmd("git branch --show-current", cwd=LOCAL_DIR)
run_cmd("git status --short", check=False, cwd=LOCAL_DIR)


$ git ls-remote --heads https://***TOKEN***@github.com/ClarisyaA/fingo-income-analysis.git feat/income-predictor-final
db491b5df8a81f677fd9093fdb0ee4c72705060a	refs/heads/feat/income-predictor-final
$ git clone -b feat/income-predictor-final https://***TOKEN***@github.com/ClarisyaA/fingo-income-analysis.git /content/fingo-income-analysis
Cloning into '/content/fingo-income-analysis'...
$ git remote set-url origin https://***TOKEN***@github.com/ClarisyaA/fingo-income-analysis.git

Repo siap digunakan
Working directory: /content/fingo-income-analysis
$ git branch --show-current
feat/income-predictor-final
$ git status --short


CompletedProcess(args='git status --short', returncode=0, stdout='', stderr='')

In [2]:
# CELL 05.2 — Setup
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', 'scikit-learn', '--quiet'])

import os, json, warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
np.random.seed(42)
os.chdir("/content/fingo-income-analysis")

DIRECTION_THRESHOLD = 0.10

def ensure_dir(path):
    d = os.path.dirname(path) if '.' in os.path.basename(path) else path
    if d:
        os.makedirs(d, exist_ok=True)

def safe_to_csv(df, path, **kwargs):
    ensure_dir(path)
    df.to_csv(path, index=kwargs.pop('index', False), **kwargs)

def classify_direction(next_income, current_income, threshold=DIRECTION_THRESHOLD):
    if current_income <= 0:
        return 'Up' if next_income > 0 else 'Stable'
    pct = (next_income - current_income) / current_income
    if pct >= threshold:
        return 'Up'
    elif pct <= -threshold:
        return 'Down'
    return 'Stable'

ORDERED_GIG_TYPES = [
    'ojek_online', 'kurir', 'jualan_online', 'freelance_desain',
    'freelance_it', 'content_creator', 'tutor', 'pekerja_harian'
]

print('Setup selesai')


Setup selesai


In [3]:
# CELL 05.3 — Load synthetic data
df_synth_raw = pd.read_csv('data/synthetic/synthetic_52week_user_income.csv')
print(f'Dimuat: synthetic_52week_user_income.csv ({df_synth_raw.shape})')

FORBIDDEN = [
    'next_week_income', 'next_week_income_norm', 'next_week_direction',
    'monthly_income', 'avg_weekly_income', 'income_std_4w', 'income_cv_4w',
    'income_range_4w', 'income_w1', 'income_w2', 'income_w3', 'income_w4',
    'synthetic_weekly_income',
]

print('Forbidden leakage columns:')
for c in FORBIDDEN:
    print(f'  - {c}')


Dimuat: synthetic_52week_user_income.csv ((156000, 36))
Forbidden leakage columns:
  - next_week_income
  - next_week_income_norm
  - next_week_direction
  - monthly_income
  - avg_weekly_income
  - income_std_4w
  - income_cv_4w
  - income_range_4w
  - income_w1
  - income_w2
  - income_w3
  - income_w4
  - synthetic_weekly_income


In [4]:
# CELL 05.4 — Generate forecasting dataset dengan sliding window
# Urutan kronologis BENAR: income_w4 (terlama) → w3 → w2 → w1 (terbaru)
# Sliding window: dari 52-week history

synth_fc_rows = []

for syn_uid, user_df in df_synth_raw.groupby('synthetic_user_id'):
    user_df = user_df.sort_values('week_index').reset_index(drop=True)
    incomes = user_df['synthetic_weekly_income'].values
    n_weeks = len(user_df)

    if n_weeks < 5:
        continue

    for i in range(4, n_weeks):
        l4, l3, l2, l1, nxt = incomes[i-4], incomes[i-3], incomes[i-2], incomes[i-1], incomes[i]
        lv = [l4, l3, l2, l1]

        rm = np.mean(lv)
        rs = np.std(lv)
        rmin = np.min(lv)
        rmax = np.max(lv)

        ta = l1 - l2
        tp = ta / l2 if l2 > 0 else 0.0

        pd_ = classify_direction(l1, l2)
        lr = l1 / rm if rm > 0 else 1.0
        rcv = rs / rm if rm > 0 else 0.0
        rmed = float(np.median(lv))
        rlmp = (l1 - rmed) / rmed if rmed > 0 else 0.0

        try:
            sl4 = np.polyfit(range(4), lv, 1)[0]
        except Exception:
            sl4 = 0.0

        trow = user_df.iloc[i]
        gt = trow.get('gig_type', 'pekerja_harian')

        history_long = incomes[max(0, i-8):i]
        rm8 = np.mean(history_long) if len(history_long) > 0 else rm
        rs8 = np.std(history_long) if len(history_long) > 1 else rs
        rm2 = np.mean(incomes[max(0, i-2):i])

        income_growth_1w = (l1 - l2) / l2 if l2 > 0 else 0.0
        income_volatility = rs / rm if rm > 0 else 0.0

        fc = {
            'synthetic_user_id': syn_uid,
            'source_respondent_id': trow.get('source_respondent_id', ''),
            'dataset_type': 'synthetic_52w',
            'target_week_index': i + 1,
            'target_date': trow['week_start_date'],

            # TARGETS (jangan masuk FEATURE_COLS)
            'next_week_income': nxt,
            'next_week_direction': classify_direction(nxt, l1),

            # Lag features (urutan: lag_4 = terlama, lag_1 = terbaru)
            'lag_1_income': l1,
            'lag_2_income': l2,
            'lag_3_income': l3,
            'lag_4_income': l4,

            # Rolling stats 4w
            'rolling_mean_4w': rm,
            'rolling_std_4w': rs,
            'rolling_min_4w': rmin,
            'rolling_max_4w': rmax,
            'rolling_range_4w': rmax - rmin,
            'rolling_median_4w': rmed,
            'rolling_cv_4w': rcv,
            'rolling_last_vs_median_pct': rlmp,

            # Rolling stats 2w and 8w
            'rolling_mean_2w': rm2,
            'rolling_mean_8w': rm8,
            'rolling_std_8w': rs8,

            # Trend features
            'income_trend_4w_abs': ta,
            'income_trend_4w_pct': tp,
            'last_income_change_abs': ta,
            'last_income_change_pct': tp,
            'income_growth_1w': income_growth_1w,
            'income_volatility': income_volatility,
            'trend_slope_4w': sl4,

            # Direction features
            'previous_direction': pd_,
            'is_previous_week_up': int(pd_ == 'Up'),
            'is_previous_week_down': int(pd_ == 'Down'),
            'is_previous_week_stable': int(pd_ == 'Stable'),
            'lag_ratio_1_to_mean': lr,

            # Calendar features
            'target_month': trow['month'],
            'target_week_of_month': trow['week_of_month'],
            'target_quarter': trow['quarter'],
            'target_is_month_start': trow['is_month_start'],
            'target_is_month_end': trow['is_month_end'],
            'target_is_payday_period': trow['is_payday_period'],
            'target_is_weekend': trow['is_weekend'],
            'target_is_ramadan_lebaran': trow['is_ramadan_lebaran_period'],
            'target_is_harbolnas': trow['is_harbolnas_period'],
            'target_is_christmas_year_end': trow['is_christmas_year_end'],
            'target_is_new_year': trow['is_new_year'],
            'seasonal_event_type': trow['seasonal_event_type'],

            # Profile
            'gig_type': gt,
            'domisili_code': trow.get('domisili_code', 'jabodetabek'),
            'usia': trow.get('usia', 25),
            'experience_months_log': trow.get('experience_months_log', 0),
            'hari_kerja_per_minggu': trow.get('hari_kerja_per_minggu', 5),
            'jam_kerja_per_hari': trow.get('jam_kerja_per_hari', 8),
            'total_jam_seminggu': trow.get('total_jam_seminggu', 40),
            'bps_jasa_weekly': trow.get('bps_jasa_weekly', 125000),

            # Preferences
            'pref_awal_bulan': trow.get('pref_awal_bulan', 0),
            'pref_payday': trow.get('pref_payday', 0),
            'pref_weekend': trow.get('pref_weekend', 0),
            'pref_ramadan_lebaran': trow.get('pref_ramadan_lebaran', 0),
            'pref_natal_tahun_baru': trow.get('pref_natal_tahun_baru', 0),
            'pref_harbolnas': trow.get('pref_harbolnas', 0),
            'pref_promo_aplikasi': trow.get('pref_promo_aplikasi', 0),
        }

        # OHE gig_type
        for g in ORDERED_GIG_TYPES:
            fc[f'gig_{g}'] = 1 if gt == g else 0

        synth_fc_rows.append(fc)

df_income_features = pd.DataFrame(synth_fc_rows)

print(f'Features dataset: {len(df_income_features):,} rows, {df_income_features.shape[1]} cols')
print(f'Users: {df_income_features["synthetic_user_id"].nunique():,}')
print(df_income_features.head().to_string(index=False))


Features dataset: 144,000 rows, 69 cols
Users: 3,000
synthetic_user_id source_respondent_id  dataset_type  target_week_index target_date  next_week_income next_week_direction  lag_1_income  lag_2_income  lag_3_income  lag_4_income  rolling_mean_4w  rolling_std_4w  rolling_min_4w  rolling_max_4w  rolling_range_4w  rolling_median_4w  rolling_cv_4w  rolling_last_vs_median_pct  rolling_mean_2w  rolling_mean_8w  rolling_std_8w  income_trend_4w_abs  income_trend_4w_pct  last_income_change_abs  last_income_change_pct  income_growth_1w  income_volatility  trend_slope_4w previous_direction  is_previous_week_up  is_previous_week_down  is_previous_week_stable  lag_ratio_1_to_mean  target_month  target_week_of_month  target_quarter  target_is_month_start  target_is_month_end  target_is_payday_period  target_is_weekend  target_is_ramadan_lebaran  target_is_harbolnas  target_is_christmas_year_end  target_is_new_year seasonal_event_type         gig_type domisili_code  usia  experience_months_log  har

In [5]:
# CELL 05.5 — Anti-leakage check
FEATURE_COLS = [
    'lag_1_income', 'lag_2_income', 'lag_3_income', 'lag_4_income',
    'rolling_mean_4w', 'rolling_std_4w', 'rolling_min_4w', 'rolling_max_4w', 'rolling_range_4w',
    'rolling_median_4w', 'rolling_cv_4w', 'rolling_last_vs_median_pct',
    'rolling_mean_2w', 'rolling_mean_8w', 'rolling_std_8w',
    'income_trend_4w_abs', 'income_trend_4w_pct', 'last_income_change_abs', 'last_income_change_pct',
    'income_growth_1w', 'income_volatility', 'trend_slope_4w',
    'is_previous_week_up', 'is_previous_week_down', 'is_previous_week_stable',
    'lag_ratio_1_to_mean',
    'target_month', 'target_week_of_month', 'target_quarter',
    'target_is_month_start', 'target_is_month_end', 'target_is_payday_period',
    'target_is_weekend', 'target_is_ramadan_lebaran', 'target_is_harbolnas',
    'target_is_christmas_year_end', 'target_is_new_year',
    'usia', 'experience_months_log', 'hari_kerja_per_minggu', 'jam_kerja_per_hari',
    'total_jam_seminggu', 'bps_jasa_weekly',
    'pref_awal_bulan', 'pref_payday', 'pref_weekend', 'pref_ramadan_lebaran',
    'pref_natal_tahun_baru', 'pref_harbolnas', 'pref_promo_aplikasi',
] + [f'gig_{g}' for g in ORDERED_GIG_TYPES]

FEATURE_COLS = [c for c in FEATURE_COLS if c in df_income_features.columns]

leaked = [c for c in FORBIDDEN if c in FEATURE_COLS]
assert len(leaked) == 0, f'LEAKAGE DETECTED: {leaked}'

print(f'Anti-leakage PASSED — {len(FEATURE_COLS)} features')
print('Income sequence ordering: lag_4 (terlama) → lag_3 → lag_2 → lag_1 (terbaru)')

print('\nFeature columns:')
for c in FEATURE_COLS:
    print(f'  - {c}')


Anti-leakage PASSED — 58 features
Income sequence ordering: lag_4 (terlama) → lag_3 → lag_2 → lag_1 (terbaru)

Feature columns:
  - lag_1_income
  - lag_2_income
  - lag_3_income
  - lag_4_income
  - rolling_mean_4w
  - rolling_std_4w
  - rolling_min_4w
  - rolling_max_4w
  - rolling_range_4w
  - rolling_median_4w
  - rolling_cv_4w
  - rolling_last_vs_median_pct
  - rolling_mean_2w
  - rolling_mean_8w
  - rolling_std_8w
  - income_trend_4w_abs
  - income_trend_4w_pct
  - last_income_change_abs
  - last_income_change_pct
  - income_growth_1w
  - income_volatility
  - trend_slope_4w
  - is_previous_week_up
  - is_previous_week_down
  - is_previous_week_stable
  - lag_ratio_1_to_mean
  - target_month
  - target_week_of_month
  - target_quarter
  - target_is_month_start
  - target_is_month_end
  - target_is_payday_period
  - target_is_weekend
  - target_is_ramadan_lebaran
  - target_is_harbolnas
  - target_is_christmas_year_end
  - target_is_new_year
  - usia
  - experience_months_log
  - 

In [6]:
# CELL 05.6 — Simpan income_features.csv + feature list
safe_to_csv(df_income_features, 'data/processed/income_features.csv')

feature_meta = {
    'feature_columns': FEATURE_COLS,
    'target_regression': 'next_week_income',
    'target_classification': 'next_week_direction',
    'n_features': len(FEATURE_COLS),
    'income_sequence_note': 'lag_4=terlama(W4), lag_1=terbaru(W1). Urutan kronologis: lag_4→lag_3→lag_2→lag_1',
    'direction_threshold': f'{DIRECTION_THRESHOLD} (>= Up, <= Down)',
}

ensure_dir('outputs/model_contract/')

with open('outputs/model_contract/feature_columns.json', 'w', encoding='utf-8') as f:
    json.dump(feature_meta, f, indent=2, ensure_ascii=False)

print(f'Disimpan: data/processed/income_features.csv ({df_income_features.shape})')
print('Disimpan: outputs/model_contract/feature_columns.json')


Disimpan: data/processed/income_features.csv ((144000, 69))
Disimpan: outputs/model_contract/feature_columns.json


In [7]:
# CELL 05.7 — Quick validation output
print('=== income_features.csv validation ===')
print(f'Rows: {len(df_income_features):,}')
print(f'Columns: {df_income_features.shape[1]}')
print(f'Feature columns: {len(FEATURE_COLS)}')
print(f'Users: {df_income_features["synthetic_user_id"].nunique():,}')

print('\nTarget regression summary:')
print(df_income_features['next_week_income'].describe().round(0).to_string())

print('\nTarget direction distribution:')
print(df_income_features['next_week_direction'].value_counts().to_string())

print('\nTarget week range:')
print(df_income_features['target_week_index'].min(), '-', df_income_features['target_week_index'].max())


=== income_features.csv validation ===
Rows: 144,000
Columns: 69
Feature columns: 58
Users: 3,000

Target regression summary:
count     144000.0
mean      397589.0
std       303101.0
min           15.0
25%       181988.0
50%       319769.0
75%       517966.0
max      1940300.0

Target direction distribution:
next_week_direction
Stable    92587
Up        34251
Down      17162

Target week range:
5 - 52


In [8]:
# GIT PUSH — Commit dan push output notebook ini ke GitHub
import os, subprocess

LOCAL_DIR   = "/content/fingo-income-analysis"
BRANCH_NAME = "feat/income-predictor-final"
NOTEBOOK_NAME = "05_Feature_Engineering.ipynb"

os.chdir(LOCAL_DIR)

def run_cmd(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"$ {cmd}")
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f"Command gagal: {cmd}")
    return r

run_cmd('git config user.email "adelineclarisya@gmail.com"')
run_cmd('git config user.name "ClarisyaA"')

print("\n[1] Cek status")
run_cmd("git status --short", check=False)

print("\n[2] Add semua perubahan output")
run_cmd("git add data/ outputs/ notebooks/ *.ipynb", check=False)

print("\n[3] Commit")
commit_result = run_cmd(
    f'git commit -m "feat(DS2): output dari {NOTEBOOK_NAME}"',
    check=False
)
if commit_result.returncode != 0:
    print("[INFO] Tidak ada perubahan baru, skip commit.")

print("\n[4] Fetch remote terbaru")
run_cmd("git fetch origin")

print("\n[5] Rebase lalu push")
run_cmd(f"git pull --rebase origin {BRANCH_NAME}")
run_cmd(f"git push -u origin {BRANCH_NAME}")

print("\nPush berhasil!")


$ git config user.email "adelineclarisya@gmail.com"
$ git config user.name "ClarisyaA"

[1] Cek status
$ git status --short
?? data/processed/income_features.csv
?? outputs/model_contract/feature_columns.json

[2] Add semua perubahan output
$ git add data/ outputs/ notebooks/ *.ipynb

[3] Commit
$ git commit -m "feat(DS2): output dari 05_Feature_Engineering.ipynb"
[feat/income-predictor-final 36bfb76] feat(DS2): output dari 05_Feature_Engineering.ipynb
 2 files changed, 144068 insertions(+)
 create mode 100644 data/processed/income_features.csv
 create mode 100644 outputs/model_contract/feature_columns.json

[4] Fetch remote terbaru
$ git fetch origin

[5] Rebase lalu push
$ git pull --rebase origin feat/income-predictor-final
Current branch feat/income-predictor-final is up to date.
From https://github.com/ClarisyaA/fingo-income-analysis
 * branch            feat/income-predictor-final -> FETCH_HEAD
$ git push -u origin feat/income-predictor-final
Branch 'feat/income-predictor-final' 